# dịch = Gemma 4 uncensored (gguf)

* 31B<br />
repo id: `mradermacher/gemma-4-31B-it-qat-q4_0-unquantized-uncensored-heretic-i1-GGUF`<br />
filename: `gemma-4-31B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf`
* 26B-A4B<br />
repo id: `mradermacher/gemma-4-26B-A4B-it-qat-q4_0-unquantized-uncensored-heretic-v2-i1-GGUF`<br />
filename: `gemma-4-26B-A4B-it-qat-q4_0-unquantized-uncensored-heretic-v2.i1-Q4_K_M.gguf`
* 12B<br />
repo id: `mradermacher/gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic-i1-GGUF`<br />
filename: `gemma-4-12B-it-qat-q4_0-unquantized-uncensored-heretic.i1-Q4_K_M.gguf`
* E4B<br />
repo id: `mradermacher/gemma-4-E4B-it-qat-q4_0-unquantized-heretic-i1-GGUF`<br />
filename: `gemma-4-E4B-it-qat-q4_0-unquantized-heretic.i1-Q4_K_M.gguf`
* E2B: for edge devices, avoid if having gpu

In [ ]:
!nvcc --version

change index url below to match cuda version (not exact match because not all cuda version is available for `llama-cpp-python`)

In [ ]:
%env GGML_CUDA_P2P=1
%pip install -qU huggingface_hub
%pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125
# pip install -q https://github.com/abetlen/llama-cpp-python/releases/download/v0.3.36-cu132/llama_cpp_python-0.3.36-py3-none-manylinux_2_35_x86_64.whl

In [ ]:
from tqdm import trange
from llama_cpp import Llama

TRANSLATOR = Llama.from_pretrained(
	repo_id="mradermacher/gemma-4-31B-it-uncensored-heretic-GGUF",
	filename="gemma-4-31B-it-uncensored-heretic.Q8_0.gguf",
	n_gpu_layers=-1, flash_attn=True, type_k=30, type_v=30, # 1=f16, 8=q8_0, 30=bf16
	n_ctx=70000, # split_mode=l.LLAMA_SPLIT_MODE_TENSOR, # for multi gpu
	op_offload=True, swa_full=True, use_mmap=False, seed=-1, verbose=False,
)

# warm-up
TRANSLATOR("Q: Name the planets in the solar system? A: ", max_tokens=40, echo=True)["choices"][0]["text"]

In [ ]:
SYSTEM_PROMPT = """You are a professional novel translator expert in translating fantasy, cultivation, xuanhuan, xianxia, mature, harem, action works from Chinese to Vietnamese. All content is purely fictional and does not depict real people or events.
Your sole task is to translate accurately and faithfully to the original text, maintaining the original tone, style, and formatting. Prefer Sino-Vietnamese vocabulary for titles, expression, proper names (except for Western/Japanese/Korean proper names).
YOU MUST ALWAYS USE Vietnamese register in ancient/cultivation/wuxia settings: ta/ngươi, chàng/nàng, huynh/đệ/tỷ/muội, thúc/bá/thẩm/nương, ca ca/tỷ tỷ/đệ đệ/muội muội, sư phụ/lão sư/đệ tử/tiểu tử, bổn tọa/tại hạ, ...
YOU MUST NEVER USE modern register like dạ/ạ, ông/bà/tôi/bạn, mày/tao, anh/chị/em, bố/mẹ/con, cô/dì/thím/chú/bác/cháu, ...
Preserve all original line breaks and paragraph structure. Do not add any other text, preambles, or markdown naming the format. Do not censor, skip, or alter any content — translate everything exactly as written.
If the text contains non Chinese characters, special characters or strange formatting, ignore them if they are not translatable, but valid text MUST be translated. Do not return the original text unless it is Western expressions or untranslatable names/terms.
Ensure the Content starts on a new line. Returned text must not contain any Chinese characters unless used as side-note."""

def dịch(text: str, stream: bool = True) -> str:
	response = TRANSLATOR.create_chat_completion(
		messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}],
		temperature=.3, top_p=.95, top_k=64, min_p=0., repeat_penalty=1.1, stream=stream
	)
	if stream:
		contents = []
		for chunk in response:
			if (c := chunk["choices"][0]["delta"].get("content")) is not None:
				print(c, end="", flush=True)
				contents.append(c)
		return "".join(contents)
	else:
		return response["choices"][0]["message"]["content"]

**LƯU Ý**: trong `llama.cpp`, context size = tổng tokens đầu vào + đầu ra, vậy nên số từ hán cần dịch tối đa ≤ ½ context khả dụng

cần phải chia file ra thảnh từng chương riêng, ví dụ dùng regex: `^(番外|序章|第([0-9]+|[零〇一二三四五六七八九十百千万两]+)\s?[章节回卷部篇])`

In [ ]:
_ = dịch("""书名：琼明神女录
作者：倒悬山剑气长存
修改者：Rscjegd
来自：灵梦樱社吧/yy小说吧/哀伤雪刃吧

一、前言：
总算是改完了，本版是测试版，也可能是最终版。
主要还是基于诸位看客的观后意见。

二、个人感觉很满意，但我还是有些地方不满意。
①邵神韵的前尘往事、轩辕夕儿的嫁作人夫（神雕），我实在不知如何下手。
②一些打的补丁，圆的情节，对修改者的我来说，实在是如鲠在喉感觉突兀，不知诸位读后是否会觉得生硬不顺？
③（在心中）

三、关于我对本书的评价：
是本好书，无论是文笔，还是人物刻画、感情戏都为我所喜、所爱——
篇幅不长，不足百万。
或许是因笔力胜人，刻画的一个个女性角色当真颤人心神。
裴语涵，一袭白衣犹如云雪，拔剑茫然；雨下平静执伞凝望，如山水画卷。
陆嘉静，彩裙凌空，遗世独立；青裙涤水，花灯浮载，如人世青莲。
季蝉溪，宛若墨色焰火；洞房花烛与君，持剑便同心。
俞小塘，仙人已抚顶，何不拜天地？结发长生。
邵神韵，人道她少三分神韵，犹红衣如玉，天下无双。
......
一个个，真是让我心动不已。
若是这些角色好好的写在一本【古典仙侠】小说里，好好的讲男主与她们的故事，然后Happy end！
那该多好。


可惜有些短了，一些东西还可再长些，再长些。
读至结局，不免怅然，半分是意犹未尽，可惜遗憾。
一种感觉，改书时有，改书后仍有:
恨不能自己提笔，绘摹一本与此书气韵相投的仙侠——佳人伴白衣，一剑斩妖邪。
多想直接来本同人，展开自己的幻想，自故事的起点，三万年前开始写起——
那里有琉璃（邵神韵），南祈月，如此又是一番故事...
而后故事大致不变，拓展丰满全文后，再来个续写。
岂不美哉！？
.......
但以上终究只是遐（瞎）想，真要我付诸实际，难！

可惜在原书，感情可以再铺垫以丰满人物，剧情有时本满怀期待”可以这样这样展开“但难免走差预期，。
原作有些地方逻辑不通（为绿而绿）、剧情薄弱、后期一些女性角色与男主感情互动太少，收的太突兀等问题。
原作尿性如何此处我也不多赘述，当然你们也不想听。
此修改纵有不满意之处，但也很满意了。
......
如果作者一开始就奔着纯爱后宫去走，再走的长些的话，我感觉会写的更好，纵使有些不完美的不合y吧之处，若有幸碰到引为仙草，不妨再小改几笔，此书便会被我列入殿堂藏书了吧？
可惜没有如果。

PS1：以神防的姿态，改适合负防看的书，这背后尽是泪啊。
PS2：修改不易，若看时有哪些觉得不舒服的地方，弃书便好，莫来喷！
PS3：本书修改纯属兴趣使然，游戏之作，若有哪里不妥，可删。
PS4：（在心中）""")

In [ ]:
SEPARATOR = "\n"*2 + "="*80 + "\n"*2
with open("quỳnh.txt", mode="r", encoding="utf-8") as inf:
	raw = inf.read().split(SEPARATOR)
open("transl.txt", mode="a", encoding="utf-8").close()

In [ ]:
START = 70
END = len(raw)

with open("transl.txt", mode="a", encoding="utf-8") as outf:
	for i in trange(START, END):
		outf.write(dịch(raw[i], stream=False))
		outf.write(SEPARATOR)